In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

In [ ]:
import anndata as ad
from adjustText import adjust_text

from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import cellrank as cr
from cellrank.estimators import GPCCA

import doubletdetection

from fa2 import ForceAtlas2

import gc

import harmonypy as hm

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

import networkx as nx

import numpy as np

import palantir

import pandas as pd

import phate

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk
import os, subprocess
from matplotlib.collections import PathCollection

In [ ]:
import sys

sys.path.append('..')

from pyfuncs.io import save_deg_to_excel_simple, load_full_adata, add_ensembl_ids
from pyfuncs.dropletQC import classify_empty_and_damaged
from pyfuncs.general import preprocessing_adata_sub
from pyfuncs.plot_functions import magma, set_plotting_style, plot_volcano, plot_cell_stats, plot_gene_stats, savefig
from pyfuncs.common_vars import BASE_DIR, SEED, CELLBENDER_FIXED_ARGS
set_plotting_style()

In [ ]:
from pyfuncs.qc import  MT_CONTIG_MOUSE_REFSEQ, compute_qc_metrics, add_droplet_qc, flag_doublets, qc_embedding, plot_qc_overview, nf_band_report, ambient_top_genes, apply_qc_flags, qc_summary
from pyfuncs.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report
from pyfuncs.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report
from pyfuncs.characterization import subset_and_reprocess, check_marker_dict, population_composition, reprocess_in_place
from pyfuncs.cell_types import DICT_MARKERS_MAJOR_POPULATIONS, DICT_MARKERS_FAP, DICT_MARKERS_KRANOCYTE, DICT_MARKERS_SATELLITE, DICT_MARKERS_TENO, DICT_RENAMING, PALETTE_CELL_TYPE
from pyfuncs.ontology import build_background, background_sensitivity, top_degs, run_goea, collapse_redundant, plot_goea

In [ ]:
from pyfuncs.io import ambient_fraction_per_gene
from pyfuncs.trajectory import (
    normalize_velocyto_layers,
    run_velocity,
    paga_report,
    paga_stability,
    plot_velocity,
    run_phate
)

In [ ]:
from datetime import date
TODAY = str(date.today())

DATA_DIR = f"{BASE_DIR}/data/public_scRNAseq/datasets/oprescu_2020_GSE138826"
FIG_DIR = f"{BASE_DIR}/figures/{TODAY}/"
RESULTS_DIR = f"{BASE_DIR}/results/{TODAY}/"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

CELL_TYPE="cell_type"
SEED = 10

In [ ]:
GRUPO = "cell_type"     # la partición sobre la que se define PAGA
LOTE  = "gsm"

# Globinas: 45% del transcriptoma en algunas muestras, y 100% spliced.
# Fuera de velocity_genes sí o sí.
HB = ["Hbb-bs", "Hba-a1", "Hba-a2", "Hbb-bt", "Alas2", "Ahsp", "Bpgm"]

# Adata loading

In [ ]:
# adata = sc.read(f"{DATA_DIR}/processed_adatas/OP_oprescu_ processed.h5ad")
adata_FAP = sc.read(f"{DATA_DIR}/processed_adatas/OP_oprescu_processed_FAPs_cleared.h5ad")

## FAPs

In [ ]:
adata_FAP_velo = adata_FAP.copy()

reprocess_in_place(adata_FAP_velo, integrate=True)         
adata_FAP_velo.var["ambient_fraction"] = ambient_fraction_per_gene(adata_FAP_velo)

normalize_velocyto_layers(adata_FAP_velo, verbose=True)
run_velocity(adata_FAP_velo, use_rep="X_harmony", normalize_su=False, correct_su=False,
             ambient_max=0.2, n_jobs=20, mode="stochastic")

adata_FAP_velo.var.loc[adata_FAP_velo.var_names.isin(HB), "velocity_genes"] = False
print("genes de velocity tras quitar globinas:", int(adata_FAP_velo.var["velocity_genes"].sum()))

# velocity_genes cambió, así que el grafo y el pseudotiempo hay que rehacerlos
scv.tl.velocity_graph(adata_FAP_velo, n_jobs=20)
scv.tl.velocity_pseudotime(adata_FAP_velo)
scv.tl.velocity_confidence(adata_FAP_velo)
print("confianza media:", round(float(adata_FAP_velo.obs["velocity_confidence"].mean()), 3))


In [ ]:
DICT_RENAME_DAYS = {
    "non_injured": "NI",
    "0.5_dpi": "D0.5",
    "2_dpi": "D2",
    "3.5_dpi": "D3.5",
    "5_dpi": "D5",
    "10_dpi": "D10",
    "21_dpi": "D21"
}

adata_FAP_velo.obs["day"] = adata_FAP_velo.obs["condition"].map(DICT_RENAME_DAYS).astype("category")
adata_FAP_velo.uns["day_colors"] = ["#ffdb00", "#ffa904", "#ee7b06", "#a12424", "#8A2664", "#5F1C86", "#26379B"]


In [ ]:
adata_FAP_velo.obs[GRUPO] = adata_FAP_velo.obs[GRUPO].astype("category").cat.remove_unused_categories()

rep = paga_report(adata_FAP_velo, GRUPO, use_time_prior="velocity_pseudotime")
# Si tienes una variable ordinal externa (día, estadio, dosis, un score):
# rep = paga_report(a, GRUPO, external_key="timepoint")

aristas = rep["aristas"]
print("\nlas 8 aristas más fuertes:")
print(aristas.head(8)[["direccion", "flujo_neto", "conectividad",
                       "delta_pseudotiempo"]].round(3).to_string(index=False))

In [ ]:
adata_FAP_velo = run_phate(adata_FAP_velo, seed=SEED)

In [ ]:
fig, ax = plt.subplots(1, 1)
plot_velocity(adata_FAP_velo, basis="umap", color=GRUPO, legend_fontsize=10,
              legend_loc="right", alpha=1, s=10, arrow_size=4,
              arrow_length=10, title="1bc", ax=ax)
savefig(fig=fig, filename=f"2OP_velocyto_UMAP_FAP", fig_dir=FIG_DIR)
# El threshold es una decisión de dibujo: mira la tabla de aristas y elige uno
# que deje ver la estructura sin inventarla. Prueba varios y quédate con el que
# declares.

fig, ax = plt.subplots(1,1)

scv.pl.umap(adata_FAP_velo, color=GRUPO, ax=ax, show=False, legend_loc="right")
scv.pl.paga(adata_FAP_velo, basis="umap", color=GRUPO, size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5, threshold=0.15, 
            title=f"PAGA dirigido (threshold=0.15)", show=False, ax=ax)

node_collections = [
    c for c in ax.collections
    if isinstance(c, PathCollection)
]

node_collections[-1].set_edgecolor("black")
node_collections[-1].set_linewidth(0.5)
savefig(fig=fig, filename=f"2OP_PAGA_UMAP_FAP", fig_dir=FIG_DIR)


In [ ]:
fig, ax = plt.subplots(1, 1)
plot_velocity(adata_FAP_velo, basis="phate", color=GRUPO, legend_fontsize=10,
              legend_loc="right", alpha=1, s=10, arrow_size=4,
              arrow_length=10, title="1bc", ax=ax)
savefig(fig=fig, filename=f"2OP_velocyto_PHATE_FAP", fig_dir=FIG_DIR)

# El threshold es una decisión de dibujo: mira la tabla de aristas y elige uno
# que deje ver la estructura sin inventarla. Prueba varios y quédate con el que
# declares.

fig, ax = plt.subplots(1,1)
scv.pl.phate(adata_FAP_velo, color=GRUPO, ax=ax, show=False, legend_loc="right")
scv.pl.paga(adata_FAP_velo, basis="phate", color=GRUPO, size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5, threshold=0.15, 
            title=f"PAGA dirigido (threshold=0.15)", show=False, ax=ax)

node_collections = [
    c for c in ax.collections
    if isinstance(c, PathCollection)
]

node_collections[-1].set_edgecolor("black")
node_collections[-1].set_linewidth(0.5)
savefig(fig=fig, filename=f"2OP_PAGA_PHATE_FAP", fig_dir=FIG_DIR)


In [ ]:
fig, ax = plt.subplots(1, 1)
plot_velocity(adata_FAP_velo, basis="phate", color="day", legend_fontsize=10,
              legend_loc="right", alpha=1, s=10, arrow_size=4,
              arrow_length=10, title="1bc", ax=ax)
savefig(fig=fig, filename=f"2OP_velocyto_PHATE_FAP_day", fig_dir=FIG_DIR)

# El threshold es una decisión de dibujo: mira la tabla de aristas y elige uno
# que deje ver la estructura sin inventarla. Prueba varios y quédate con el que
# declares.

fig, ax = plt.subplots(1,1)
scv.pl.phate(adata_FAP_velo, color="day", ax=ax, show=False, legend_loc="right")
scv.tl.paga(adata_FAP_velo, groups="day", use_time_prior="velocity_pseudotime")
scv.pl.paga(adata_FAP_velo, basis="phate", color="day", size=50, alpha=.1,
            min_edge_width=2, node_size_scale=1.5, threshold=0.15, 
            title=f"PAGA dirigido (threshold=0.15)", show=False, ax=ax)

# node_collections = [
#     c for c in ax.collections
#     if isinstance(c, PathCollection)
# ]

# node_collections[-1].set_edgecolor("black")
# node_collections[-1].set_linewidth(0.5)
savefig(fig=fig, filename=f"2OP_PAGA_PHATE_FAP_day", fig_dir=FIG_DIR)


In [ ]:
sc.pl.umap(adata_FAP_velo, color=["cell_type", "condition"])

In [ ]:
sce.pl.phate(adata_FAP_velo, color=["cell_type", "condition"])

In [ ]:
sc.pl.umap(adata_FAP_velo, color=['day'], ncols=2)

fig, axs = plt.subplots(2, 4, figsize=(20, 10))
for idx, condition in enumerate(adata_FAP_velo.obs['day'].unique()):
    sc.pl.umap(adata_FAP_velo, 
               ax = axs.flatten()[idx], show=False, na_color="#bcbcbc", s=5)
    sc.pl.umap(adata_FAP_velo[adata_FAP_velo.obs['day'] == condition], 
               ax = axs.flatten()[idx], na_color="#252525", show=False, s=8)
    axs.flatten()[idx].set_title(condition)

axs.flatten()[-1].set_axis_off()

In [ ]:
fig, ax = plt.subplots(1, 1)
sce.pl.phate(adata_FAP_velo, color=['day'], ncols=2, ax=ax, show=False, frameon=False)
savefig(fig=fig, filename="2OP_PHATE_day", fig_dir=FIG_DIR)



fig, axs = plt.subplots(2, 4, figsize=(10, 5))
for idx, condition in enumerate(adata_FAP_velo.obs['day'].unique()):
    sce.pl.phate(adata_FAP_velo, 
               ax = axs.flatten()[idx], show=False, na_color="#bcbcbc", s=5, frameon=False)
    sce.pl.phate(adata_FAP_velo[adata_FAP_velo.obs['day'] == condition], 
               ax = axs.flatten()[idx], na_color="#252525", show=False, s=8, frameon=False)
    axs.flatten()[idx].set_title(condition)

axs.flatten()[-1].set_axis_off()
savefig(fig=fig, filename="2OP_PHATE_day_separate-each-day", fig_dir=FIG_DIR)

In [ ]:
df_US_stats = adata_FAP_velo.obs.groupby("day")[["n_spliced_log1p", "n_unspliced_log1p", "nuclear_frac"]].median()
df_US_stats["US_diff"] = df_US_stats["n_unspliced_log1p"] - df_US_stats["n_spliced_log1p"]
df_US_stats

In [ ]:
adata_FAP_velo.obs.groupby("day")[["cellbender_removed_frac", "total_counts", "pct_counts_mt"]].median()

In [ ]:
dict_DEGs_day = {}
list_df_DEGs_day = []

N_DEGS = 150

sc.tl.rank_genes_groups(adata_FAP_velo, groupby='day', use_raw=False, method='wilcoxon', pts=True, n_genes=adata_FAP_velo.shape[1])

for condition in adata_FAP_velo.obs['day'].cat.categories:
    display(condition)
    df_pvals = plot_volcano(adata_FAP_velo, condition, pval_threshold=1e-10, lfc_threshold=0.5, 
                            topn=30, bottomn=0, return_df=True, xlim=(0, 8))
    
    dict_DEGs_day[condition] = df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:N_DEGS]
    
    sce.pl.phate(adata_FAP_velo, color=df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:10], cmap=magma, use_raw=False, ncols=5)
    
    fig, axs = plt.subplots(2, 5, figsize=(20, 4))
    for idx, gene in enumerate(df_pvals.sort_values(by='pvalxlfc', ascending=False)['gene'].values[:10]):
        sc.pl.violin(adata_FAP_velo, keys=gene, 
                    groupby='day', use_raw=False, ax=axs.flatten()[idx], show=False)
    plt.tight_layout()
    plt.show()

    df_pvals['day'] = condition
    list_df_DEGs_day.append(df_pvals.sort_values(by='pvalxlfc', ascending=False))


df_pvals_all = pd.concat(list_df_DEGs_day, axis=0, ignore_index=True)
save_deg_to_excel_simple(df_pvals_all, f"{RESULTS_DIR}/2OP_df_DEGs.xlsx", group_col='day', 
                         cols=('gene', 'adj_pval', 'logfoldchanges', 'neg_log_pval', 'pvalxlfc'))

In [ ]:
def plot_jaccard(dict_rows, dict_cols):
    df_jaccard = pd.DataFrame(index = dict_rows.keys(), columns = dict_cols.keys(), dtype=np.float32)

    for row, genes_row in dict_rows.items():
        for col, genes_col in dict_cols.items():
            if row != col:
                jac = 100 * len(np.intersect1d(genes_row, genes_col)) / len(np.union1d(genes_row, genes_col))
                df_jaccard.loc[row, col] = jac
            else:
                df_jaccard.loc[row, col] = np.nan

    return df_jaccard

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
sns.heatmap(plot_jaccard(dict_DEGs_day, dict_DEGs_day), annot=True, cmap='Blues', ax=ax)
savefig(fig=fig, filename=f"2OP_jaccard_heatmap_DEGs_day_top{N_DEGS}", fig_dir=FIG_DIR)

In [ ]:
dict_DEGs_day_plot = {
                 "D0.5": ["Ccl7", "Gnl3"], 
                 "D2": ["Timp1", "Ccl2"], 
                 "D3.5": ["Postn", "Angptl4"], 
                 "D5": ["Cthrc1", "Ptn"], 
                 "D10": ["Mgp", "Col14a1"], 
                 "D21": ["Itih5", "Clec3b"], 
                 "NI": ["Pla1a", "Hsd11b1"], }


fig, axs = plt.subplots(7, 2, figsize=(10, 14))
for idx, (cond, genes) in enumerate(dict_DEGs_day_plot.items()):
    sc.pl.violin(adata_FAP_velo, keys=genes[0], 
                        groupby='day', use_raw=False, ax=axs[idx, 0], show=False)
    sc.pl.violin(adata_FAP_velo, keys=genes[1], 
                        groupby='day', use_raw=False, ax=axs[idx, 1], show=False)        
plt.tight_layout()
savefig(fig=fig, filename="2OP_violin_plot_markers_day", fig_dir=FIG_DIR)

### GO of DEGs of each day

In [ ]:
background_genes = build_background(adata_FAP_velo, min_cells=5, group_key="day")

degs = top_degs(adata_FAP_velo, "day", n_top=N_DEGS, recompute=True)

In [ ]:
res = run_goea(degs, background_genes, )
res["log_padj"] = -np.log10(res["padj_global"])

In [ ]:
fig, ax = plot_goea(res, n_top=8)
savefig(fig=fig, filename="2OP_GO-terms_top-shared_day", fig_dir=FIG_DIR)

In [ ]:
for poblacion in adata_FAP_velo.obs['day'].cat.categories:
    df_sub = res[res["poblacion"] == poblacion]
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    sns.barplot(data = df_sub.iloc[:20], y="termino", x="log_padj", ax=ax)
    plt.axvline(-np.log10(0.05), c="#bc0000")
    plt.title(f"Top GO terms for {poblacion}")
    savefig(fig=fig, filename=f"2OP_GO-terms_{poblacion}", fig_dir=FIG_DIR)
    plt.show()

In [ ]:
save_deg_to_excel_simple(res, f"{RESULTS_DIR}/2OP_df_GOEA_days.xlsx", group_col='poblacion',
    cols=('libreria','termino','solapamiento','odds_ratio', "pval", "padj_global", "genes"))